# Retail Enterprise BI Control Tower
## Notebook 01: Data Inspection

**Project:** Retail Enterprise BI Control Tower  
**Purpose:** Profile all 8 raw tables before any cleaning. Documents row counts, column names, data types, null values, duplicates, and sample categorical values. This is the official starting condition of the data.  
**Author:** Arun Prabakar Vadaseri Rajendran

**Date:** June 2026  
**Tool:** Google Colab

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import os

# Suppress display truncation so we see full column lists
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)
pd.set_option('display.width', 120)

print("Libraries loaded ✓")

Libraries loaded ✓


In [ ]:
import pandas as pd

# --- 1. Load DataFrames ---
# Define file paths
base_path = '/content/'
files = {
    'customers': 'customers.csv',
    'inventory': 'inventory.csv',
    'marketing_campaigns': 'marketing_campaigns.csv',
    'order_details': 'order_details.csv',
    'orders': 'orders.csv',
    'products': 'products.csv',
    'returns': 'returns.csv',
    'stores': 'stores.csv'
}

# Load into variables
df_customers = pd.read_csv(f"{base_path}{files['customers']}")
df_inventory = pd.read_csv(f"{base_path}{files['inventory']}")
df_marketing = pd.read_csv(f"{base_path}{files['marketing_campaigns']}")
df_order_details = pd.read_csv(f"{base_path}{files['order_details']}")
df_orders = pd.read_csv(f"{base_path}{files['orders']}")
df_products = pd.read_csv(f"{base_path}{files['products']}")
df_returns = pd.read_csv(f"{base_path}{files['returns']}")
df_stores = pd.read_csv(f"{base_path}{files['stores']}")

# --- 2. Define Table Descriptions and Summaries ---
tables = [
    ("Customers", df_customers, "Contains unique customer profiles, contact info, and demographics."),
    ("Inventory", df_inventory, "Tracks stock levels across different warehouse or store locations."),
    ("Marketing Campaigns", df_marketing, "Details on promotional events, budgets, and targeted segments."),
    ("Orders", df_orders, "High-level transaction records including dates, customer IDs, and totals."),
    ("Order Details", df_order_details, "Granular line-items for each order (product IDs, quantities, prices)."),
    ("Products", df_products, "The product catalog including names, categories, and cost prices."),
    ("Returns", df_returns, "Logs of product returns linked to specific orders for reverse logistics analysis."),
    ("Stores", df_stores, "Information about physical locations, store size, and region.")
]

# --- 3. Print Summaries ---
print(f"{'TABLE NAME':<20} | {'ROWS':>8} | {'COLS':>5} | {'COLUMNS'}")
print("-" * 100)

for name, df, desc in tables:
    # Print business context
    print(f"# {name}: {desc}")
    # Print statistics
    cols_list = ", ".join(df.columns)
    print(f"{name:<20} | {len(df):>8,} | {len(df.columns):>5} | {cols_list}")
    print("-" * 100)

print("\nAll data successfully loaded into DataFrames: df_customers, df_inventory, df_marketing, df_order_details, df_orders, df_products, df_returns, df_stores")

TABLE NAME           |     ROWS |  COLS | COLUMNS
----------------------------------------------------------------------------------------------------
# Customers: Contains unique customer profiles, contact info, and demographics.
Customers            |   50,100 |     9 | customer_id, customer_name, segment, region, city, preferred_contact, signup_date, age, income_band
----------------------------------------------------------------------------------------------------
# Inventory: Tracks stock levels across different warehouse or store locations.
Inventory            |  720,000 |     7 | store_id, product_id, snapshot_month, stock_on_hand, reorder_point, warehouse, stockout_risk
----------------------------------------------------------------------------------------------------
# Marketing Campaigns: Details on promotional events, budgets, and targeted segments.
Marketing Campaigns  |      900 |     8 | campaign_id, campaign_name, campaign_channel, target_segment, start_date, campaign

In [ ]:
import pandas as pd

# List of (name, dataframe) for iteration
df_list = [
    ("Customers", df_customers),
    ("Inventory", df_inventory),
    ("Marketing Campaigns", df_marketing),
    ("Orders", df_orders),
    ("Order Details", df_order_details),
    ("Products", df_products),
    ("Returns", df_returns),
    ("Stores", df_stores)
]

# Tracking for the final summary
issue_tracker = []

print("=== NULL VALUE INSPECTION ===\n")

# BI Context: Nulls in primary/foreign keys can lead to 'orphan' records during joins,
# while nulls in metrics (like price/quantity) can skew descriptive statistics.

for name, df in df_list:
    print(f"--- Table: {name} ---")

    # Calculate nulls
    null_counts = df.isnull().sum()
    null_pct = (df.isnull().sum() / len(df)) * 100

    # Filter to only columns with missing values
    missing_data = pd.concat([null_counts, null_pct], axis=1, keys=['Count', 'Percentage (%)'])
    missing_data = missing_data[missing_data['Count'] > 0]

    if not missing_data.empty:
        display(missing_data.style.format({'Percentage (%)': '{:.2f}%'}))
        issue_tracker.append((name, len(missing_data)))
    else:
        print("No missing values found.")

    print("\n")

# --- Final Summary ---
print("=== FINAL DATA QUALITY SUMMARY ===")
if issue_tracker:
    print(f"{'Table Name':<20} | {'Affected Columns'}")
    print("-" * 40)
    for table_name, count in issue_tracker:
        print(f"{table_name:<20} | {count} columns have nulls")
else:
    print("Perfect data quality! No null values found across all 8 tables.")

=== NULL VALUE INSPECTION ===

--- Table: Customers ---


,Count,Percentage (%)
preferred_contact,5484,10.95%




--- Table: Inventory ---
No missing values found.


--- Table: Marketing Campaigns ---
No missing values found.


--- Table: Orders ---


,Count,Percentage (%)
campaign_id,87860,67.45%




--- Table: Order Details ---


,Count,Percentage (%)
discount_pct,1212,0.40%




--- Table: Products ---
No missing values found.


--- Table: Returns ---
No missing values found.


--- Table: Stores ---
No missing values found.


=== FINAL DATA QUALITY SUMMARY ===
Table Name           | Affected Columns
----------------------------------------
Customers            | 1 columns have nulls
Orders               | 1 columns have nulls
Order Details        | 1 columns have nulls


In [ ]:
import pandas as pd

# Define configuration for duplicate checks
# Note: Inventory does not have a single unique key (it's usually a composite of store+product+month)
data_configs = [
    ("Customers", df_customers, "customer_id"),
    ("Inventory", df_inventory, None),
    ("Marketing Campaigns", df_marketing, "campaign_id"),
    ("Orders", df_orders, "order_id"),
    ("Order Details", df_order_details, "order_line_id"),
    ("Products", df_products, "product_id"),
    ("Returns", df_returns, "return_id"),
    ("Stores", df_stores, "store_id")
]

dedupe_list = []

print("=== DUPLICATION INSPECTION ===\n")

for name, df, pk in data_configs:
    print(f"--- Table: {name} ---")

    # 1. Check for fully duplicate rows
    full_dupes_count = df.duplicated().sum()
    print(f"Fully duplicate rows: {full_dupes_count}")

    # 2. Check for duplicate Primary Keys (if applicable)
    pk_dupes_count = 0
    if pk:
        pk_dupes_count = df.duplicated(subset=[pk]).sum()
        print(f"Duplicate Primary Keys ({pk}): {pk_dupes_count}")
    else:
        print("Primary Key check: N/A (Composite key table)")

    # 3. Visual sample if issues exist
    if full_dupes_count > 0 or pk_dupes_count > 0:
        dedupe_list.append(name)
        print("\nSample Duplicates:")
        # If PK dupes exist, show those as they are often more critical
        check_col = [pk] if pk else df.columns.tolist()
        display(df[df.duplicated(subset=check_col, keep=False)].head(2))

    print("\n" + "="*40 + "\n")

# --- Final Summary ---
print("=== DEDUPLICATION SUMMARY ===")
if dedupe_list:
    print(f"The following tables require deduplication logic: {', '.join(dedupe_list)}")
    print("Action: Use .drop_duplicates() in the cleaning phase to ensure metric accuracy.")
else:
    print("No duplicate issues found. Primary key integrity is intact across all tables.")

=== DUPLICATION INSPECTION ===

--- Table: Customers ---
Fully duplicate rows: 100
Duplicate Primary Keys (customer_id): 100

Sample Duplicates:


,customer_id,customer_name,segment,region,city,preferred_contact,signup_date,age,income_band
279,CU000280,Noah Martin,Small Business,Manitoba,Brandon,Email,2022-09-10,66,Medium
366,CU000367,Maya Thomas,Consumer,New Brunswick,Moncton,SMS,2020-12-10,57,High




--- Table: Inventory ---
Fully duplicate rows: 0
Primary Key check: N/A (Composite key table)


--- Table: Marketing Campaigns ---
Fully duplicate rows: 0
Duplicate Primary Keys (campaign_id): 0


--- Table: Orders ---
Fully duplicate rows: 250
Duplicate Primary Keys (order_id): 250

Sample Duplicates:


,order_id,customer_id,store_id,sales_channel,order_datetime,campaign_id,payment_method,fulfillment_type
717,OR0000718,CU017614,ONLINE,Marketplace,2025-08-11 21:14:00,CA00706,Cash,Express
1498,OR0001499,CU042136,ONLINE,Online,2024-07-28 08:35:00,UNKNOWN,Credit Card,Pickup




--- Table: Order Details ---
Fully duplicate rows: 0
Duplicate Primary Keys (order_line_id): 0


--- Table: Products ---
Fully duplicate rows: 0
Duplicate Primary Keys (product_id): 0


--- Table: Returns ---
Fully duplicate rows: 0
Duplicate Primary Keys (return_id): 0


--- Table: Stores ---
Fully duplicate rows: 0
Duplicate Primary Keys (store_id): 0


=== DEDUPLICATION SUMMARY ===
The following tables require deduplication logic: Customers, Orders
Action: Use .drop_duplicates() in the cleaning phase to ensure metric accuracy.


In [ ]:
import pandas as pd

def verify_id_uniqueness(df, id_column, table_name):
    print(f"--- {table_name} ID Verification ---")
    total_rows = len(df)
    unique_ids = df[id_column].nunique()
    diff = total_rows - unique_ids

    print(f"Total Row Count: {total_rows:,}")
    print(f"Unique {id_column} Count: {unique_ids:,}")
    print(f"Difference (Duplicates): {diff:,}")

    if diff > 0:
        # Find IDs that appear more than once
        counts = df[id_column].value_counts()
        duplicate_ids = counts[counts > 1].index

        # Filter DataFrame to show these specific IDs
        samples = df[df[id_column].isin(duplicate_ids)].sort_values(by=id_column)
        print(f"\nSample rows with duplicate {id_column}:")
        display(samples.head(3))
    else:
        print(f"\nConfirmed: All {id_column} values are unique.")
    print("\n" + "="*40 + "\n")

# Run verification for Orders and Customers
verify_id_uniqueness(df_orders, 'order_id', 'Orders')
verify_id_uniqueness(df_customers, 'customer_id', 'Customers')

--- Orders ID Verification ---
Total Row Count: 130,250
Unique order_id Count: 130,000
Difference (Duplicates): 250

Sample rows with duplicate order_id:


,order_id,customer_id,store_id,sales_channel,order_datetime,campaign_id,payment_method,fulfillment_type
717,OR0000718,CU017614,ONLINE,Marketplace,2025-08-11 21:14:00,CA00706,Cash,Express
130140,OR0000718,CU017614,ONLINE,Marketplace,2025-08-11 21:14:00,CA00706,Cash,Express
130152,OR0001499,CU042136,ONLINE,Online,2024-07-28 08:35:00,UNKNOWN,Credit Card,Pickup




--- Customers ID Verification ---
Total Row Count: 50,100
Unique customer_id Count: 50,000
Difference (Duplicates): 100

Sample rows with duplicate customer_id:


,customer_id,customer_name,segment,region,city,preferred_contact,signup_date,age,income_band
279,CU000280,Noah Martin,Small Business,Manitoba,Brandon,Email,2022-09-10,66,Medium
50090,CU000280,Noah Martin,Small Business,Manitoba,Brandon,Email,2022-09-10,66,Medium
50013,CU000367,Maya Thomas,Consumer,New Brunswick,Moncton,SMS,2020-12-10,57,High


In [11]:
import pandas as pd

# List of DataFrames to inspect
df_list = [
    ("Customers", df_customers),
    ("Inventory", df_inventory),
    ("Marketing Campaigns", df_marketing),
    ("Orders", df_orders),
    ("Order Details", df_order_details),
    ("Products", df_products),
    ("Returns", df_returns),
    ("Stores", df_stores)
]

inconsistency_tracker = []

print("=== CATEGORICAL INCONSISTENCY INSPECTION ===\n")

for name, df in df_list:
    print(f"--- Table: {name} ---")

    # 1. Identify Text/Object columns
    text_cols = df.select_dtypes(include=['object']).columns

    for col in text_cols:
        # Clean local copies for comparison
        unique_raw = df[col].dropna().unique()
        unique_norm = df[col].dropna().astype(str).str.strip().str.upper().unique()

        # 2. Display unique values or count
        if len(unique_raw) <= 20:
            print(f"  Column '{col}': {list(unique_raw)}")
        else:
            print(f"  Column '{col}': {len(unique_raw)} unique values (Too many to list)")

        # 3. Flag inconsistencies (Casing and Whitespace)
        # If normalized count is less than raw count, there are duplicates hidden by case/space
        if len(unique_norm) < len(unique_raw):
            print(f"    [!] FLAG: Inconsistent formatting detected in '{col}'")
            inconsistency_tracker.append((name, col))

    print("\n")

# 4. Final Summary
print("=== STANDARDIZATION SUMMARY ===")
if inconsistency_tracker:
    print(f"{'Table':<20} | {'Column to Standardize'}")
    print("-" * 50)
    for table, col in inconsistency_tracker:
        print(f"{table:<20} | {col}")

    print("\nBI IMPACT: Inconsistent strings lead to 'Split Groups' in Power BI/Tableau.")
    print("Example: 'Canada' and ' canada' will appear as two separate bars in a chart, skewing metrics.")
else:
    print("No categorical inconsistencies found. Text data is uniform.")

=== CATEGORICAL INCONSISTENCY INSPECTION ===

--- Table: Customers ---
  Column 'customer_id': 50000 unique values (Too many to list)
  Column 'customer_name': 360 unique values (Too many to list)
  Column 'segment': ['Student', 'Corporate', 'Consumer', 'Small Business', 'Premium']
  Column 'region': ['Quebec', 'New Brunswick', 'Alberta', 'British Columbia', 'Ontario', 'Nova Scotia', 'Manitoba', 'Saskatchewan']
  Column 'city': 30 unique values (Too many to list)
  Column 'preferred_contact': ['SMS', 'Email', 'Push']
  Column 'signup_date': 2550 unique values (Too many to list)
  Column 'income_band': ['Medium', 'High', 'Low']


--- Table: Inventory ---
  Column 'store_id': 120 unique values (Too many to list)
  Column 'product_id': 1800 unique values (Too many to list)
  Column 'snapshot_month': ['2025-01-01', '2025-02-01', '2025-03-01', '2025-04-01', '2025-05-01', '2025-06-01', '2025-07-01', '2025-08-01', '2025-09-01', '2025-10-01', '2025-11-01', '2025-12-01']
  Column 'warehouse': [

In [12]:
import pandas as pd

# Data from previous steps is stored in:
# issue_tracker: (table_name, column_count_with_nulls)
# dedupe_list: list of table names with duplicates
# inconsistency_tracker: (table_name, column_name)

report_data = []

# 1. Process Nulls (using existing issue_tracker logic)
# For the final report, we'll quickly pull the specific counts from the DFs in memory
for table_name, _ in issue_tracker:
    df_ref = next(item[1] for item in df_list if item[0] == table_name)
    null_series = df_ref.isnull().sum()
    for col, count in null_series[null_series > 0].items():
        report_data.append({
            'Table': table_name,
            'Rows': len(df_ref),
            'Issue Type': 'Null Values',
            'Column Affected': col,
            'Count': count,
            'Action Required': f"Fill {count:,} nulls with 'UNKNOWN'"
        })

# 2. Process Duplicates
for table_name in dedupe_list:
    df_ref = next(item[1] for item in df_list if item[0] == table_name)
    # Check row duplication count
    dupe_count = df_ref.duplicated().sum()
    if dupe_count > 0:
        report_data.append({
            'Table': table_name,
            'Rows': len(df_ref),
            'Issue Type': 'Duplicates',
            'Column Affected': 'All (Row level)',
            'Count': dupe_count,
            'Action Required': f"Remove {dupe_count:,} duplicate rows"
        })

# 3. Process Categorical Inconsistencies
for table_name, col in inconsistency_tracker:
    df_ref = next(item[1] for item in df_list if item[0] == table_name)
    report_data.append({
        'Table': table_name,
        'Rows': len(df_ref),
        'Issue Type': 'Category Mismatch',
        'Column Affected': col,
        'Count': 'N/A (Casing/Space)',
        'Action Required': "Standardise to title case and trim spaces"
    })

# 4. Handle Clean Tables (None)
all_table_names = [t[0] for t in df_list]
problem_tables = set([r['Table'] for r in report_data])
clean_tables = set(all_table_names) - problem_tables

for table_name in clean_tables:
    df_ref = next(item[1] for item in df_list if item[0] == table_name)
    report_data.append({
        'Table': table_name,
        'Rows': len(df_ref),
        'Issue Type': 'None',
        'Column Affected': '-',
        'Count': 0,
        'Action Required': "No action needed"
    })

# Create DataFrame for display
df_dq_report = pd.DataFrame(report_data).sort_values(by=['Issue Type', 'Table'], ascending=[False, True])

print("=== DATA QUALITY REPORT — PRE-CLEANING BASELINE ===\n")
display(df_dq_report.style.set_properties(**{'text-align': 'left'}).hide(axis='index'))

=== DATA QUALITY REPORT — PRE-CLEANING BASELINE ===



Table,Rows,Issue Type,Column Affected,Count,Action Required
Customers,50100,Null Values,preferred_contact,5484,"Fill 5,484 nulls with 'UNKNOWN'"
Order Details,303124,Null Values,discount_pct,1212,"Fill 1,212 nulls with 'UNKNOWN'"
Orders,130250,Null Values,campaign_id,87860,"Fill 87,860 nulls with 'UNKNOWN'"
Inventory,720000,None,-,0,No action needed
Marketing Campaigns,900,None,-,0,No action needed
Returns,22734,None,-,0,No action needed
Stores,120,None,-,0,No action needed
Customers,50100,Duplicates,All (Row level),100,Remove 100 duplicate rows
Orders,130250,Duplicates,All (Row level),250,Remove 250 duplicate rows
Products,1800,Category Mismatch,category,N/A (Casing/Space),Standardise to title case and trim spaces
